# Cell-Specific LDSC

Goal: estimate cell-specific enrichment by running partitioned LDSC with one query annotation per cell type.

This notebook uses a tiny synthetic dataset so every cell can run as a smoke test. For real analysis, build query annotations from cell-type BED files or `.annot.gz` files, compute baseline-plus-query LD scores, and run `partitioned-h2`.

In [ ]:
from pathlib import Path
import sys

def find_src_dir(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "ldsc").exists():
            return candidate / "src"
        nested = candidate / "ldsc_py3_restructured" / "src"
        if (nested / "ldsc").exists():
            return nested
    raise FileNotFoundError("Could not find src/ldsc from the current working directory.")

SRC_DIR = find_src_dir(Path.cwd())
REPO_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(SRC_DIR)

In [ ]:
import gzip
import os
import subprocess
import tempfile

import numpy as np
import pandas as pd

from ldsc import GlobalConfig, RegressionConfig, RegressionRunner, set_global_config
from ldsc.annotation_builder import AnnotationBundle
from ldsc.ldscore_calculator import LDScoreResult
from ldsc.sumstats_munger import SumstatsTable

## Python API

Cell-specific LDSC is represented as partitioned h2 over query annotation columns. The runner tests each query column in a baseline-plus-one-query model.

In [ ]:
GLOBAL_CONFIG = GlobalConfig(snp_identifier="rsid", genome_build="hg38")
set_global_config(GLOBAL_CONFIG)

n_snps = 20
snps = [f"rs{i}" for i in range(1, n_snps + 1)]
index = np.arange(n_snps)

sumstats = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "Z": np.linspace(-2.0, 2.0, n_snps) + 0.1 * np.sin(index),
            "N": np.full(n_snps, 10000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait",
    trait_name="trait",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

ldscore_table = pd.DataFrame(
    {
        "CHR": ["1"] * n_snps,
        "SNP": snps,
        "BP": np.arange(100, 100 + n_snps),
        "base": np.linspace(1.0, 2.0, n_snps),
        "neuron": np.linspace(0.2, 1.7, n_snps),
        "glia": np.linspace(1.5, 0.3, n_snps),
        "regr_weight": np.linspace(1.1, 1.5, n_snps),
    }
)
ldscore_result = LDScoreResult(
    ldscore_table=ldscore_table,
    snp_count_totals={
        "common_reference_snp_counts_maf_gt_0_05": np.array([100.0, 30.0, 40.0]),
        "all_reference_snp_counts": np.array([120.0, 50.0, 60.0]),
    },
    baseline_columns=["base"],
    query_columns=["neuron", "glia"],
    ld_reference_snps=frozenset(snps),
    ld_regression_snps=frozenset(snps),
    chromosome_results=[],
    config_snapshot=GLOBAL_CONFIG,
)

annotation_bundle = AnnotationBundle(
    metadata=pd.DataFrame(
        {
            "CHR": ["1"] * n_snps,
            "BP": np.arange(100, 100 + n_snps),
            "SNP": snps,
            "CM": np.linspace(0.0, 0.2, n_snps),
        }
    ),
    baseline_annotations=pd.DataFrame({"base": np.ones(n_snps)}),
    query_annotations=pd.DataFrame(
        {
            "neuron": (index % 2 == 0).astype(int),
            "glia": (index % 2 == 1).astype(int),
        }
    ),
    baseline_columns=["base"],
    query_columns=["neuron", "glia"],
    chromosomes=["1"],
    source_summary={},
    config_snapshot=GLOBAL_CONFIG,
)

ldscore_table.head()

In [ ]:
runner = RegressionRunner(
    global_config=GLOBAL_CONFIG,
    regression_config=RegressionConfig(n_blocks=10, use_intercept=False),
)
cell_specific = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)
cell_specific

## CLI

For real inputs, compute LD scores with `ldsc ldscore --baseline-annot ... --query-annot-bed ...`, then run `ldsc partitioned-h2 --query-columns ...` or pass an annotation manifest. The cell below writes the toy tables to temporary LDSC artifacts and runs the same regression CLI path.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir_raw:
    tmpdir = Path(tmpdir_raw)
    sumstats_path = tmpdir / "trait.sumstats.gz"
    ldscore_path = tmpdir / "cell_specific.l2.ldscore.gz"
    counts_path = tmpdir / "cell_specific.l2.M_5_50"
    out_prefix = tmpdir / "cell_specific"

    with gzip.open(sumstats_path, "wt", encoding="utf-8") as handle:
        sumstats.data.to_csv(handle, sep="\t", index=False)
    with gzip.open(ldscore_path, "wt", encoding="utf-8") as handle:
        ldscore_result.ldscore_table.to_csv(handle, sep="\t", index=False)
    counts_path.write_text("100\t30\t40\n", encoding="utf-8")

    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_DIR)
    command = [
        sys.executable,
        "-m",
        "ldsc",
        "partitioned-h2",
        "--sumstats",
        str(sumstats_path),
        "--trait-name",
        "trait",
        "--ldscore",
        str(ldscore_path),
        "--counts",
        str(counts_path),
        "--count-kind",
        "m_5_50",
        "--query-columns",
        "neuron,glia",
        "--n-blocks",
        "10",
        "--no-intercept",
        "--out",
        str(out_prefix),
    ]
    completed = subprocess.run(command, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"ldsc partitioned-h2 failed with exit code {completed.returncode}")
    cli_summary = pd.read_csv(tmpdir / "cell_specific.partitioned_h2.tsv", sep="\t")

cli_summary